In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd

passenger_csv = "reduced_lusitania_simple_join_dataset.csv"
cabin_csv = "reduced_cabin_inventory.csv"

class CabinPassengerViewer:
    def __init__(self, db_file):
        self.db_file = db_file
        self.conn = sqlite3.connect(self.db_file)

        cabin_table = "cabin_inventory"
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS cabin_inventory (
                cabin_id INTEGER PRIMARY KEY,
                class TEXT,
                no_of_beds INTEGER,
                price REAL,
                ship_level TEXT
            )
            """)
        df = pd.read_csv(cabin_csv)
        # db_base = connect_to_database(db_file)
        df.to_sql(cabin_table, self.conn, if_exists="replace", index=False) 

        # connecting passenger csv to passenger table
        passenger_table = "passenger"
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS passenger (
                passenger_id INTEGER PRIMARY KEY,
                Personal_name TEXT NOT NULL,
                Family_name TEXT NOT NULL,
                Age REAL,
                Citizenship TEXT,
                Sex TEXT,
                cabin_id INTEGER,
                FOREIGN KEY (cabin_id) REFERENCES cabin_inventory(cabin_id)
            )
            """)
        p_df = pd.read_csv(passenger_csv)
        p_df.to_sql(passenger_table, self.conn, if_exists="replace", index=False)

        self.root = tk.Tk()
        self.root.title("Cabin Passenger Viewer")
        self.root.geometry("750x450")

        self.create_widgets()
        self.load_cabins()

        self.root.protocol("WM_DELETE_WINDOW", self.close_app)
        self.root.mainloop()

    def join_passenger_cabin(conn):
        query = """ SELECT p.Personal_name, p.Family_name, c.class, c.ship_level, c.price
                    FROM passenger p
                    JOIN cabin_inventory c
                    ON p.cabin_id = c.cabin_id """
        
        df = pd.read_sql(query, conn)
        print("\Passenger Cabin Details")
        print(df)

    def passengers_by_cabin(conn):
        query = """ SELECT c.cabin_id, c.class, p.personal_name, p.family_name
                    FROM cabin_inventory c
                    JOIN passenger p
                    ON c.cabin_id = p.cabin_id
                    ORDER BY c.cabin_id """

        df = pd.read_sql(query, conn)
        print("\nPassengers by Cabin")
        print(df)

    def create_widgets(self):
        
        title_label = tk.Label(
            self.root,
            text="Choose a Cabin to View Its Passengers",
            font=("Arial", 14, "bold")
        )
        title_label.pack(pady=10)

        # frame for controls
        top_frame = tk.Frame(self.root)
        top_frame.pack(pady=10)

        cabin_label = tk.Label(top_frame, text="Cabin:")
        cabin_label.grid(row=0, column=0, padx=5, pady=5)

        self.cabin_combo = ttk.Combobox(top_frame, width=25, state="readonly")
        self.cabin_combo.grid(row=0, column=1, padx=5, pady=5)

        show_button = tk.Button(
            top_frame,
            text="Show Passengers",
            command=self.show_passengers
        )
        show_button.grid(row=0, column=2, padx=10, pady=5)

        # cabin details label
        self.details_label = tk.Label(
            self.root,
            text="Cabin details will appear here",
            font=("Arial", 11),
            fg="blue"
        )
        self.details_label.pack(pady=5)

        # treeview for passengers
        columns = ("passenger_id", "Personal_name", "Family_name", "Age", "Sex")
        self.tree = ttk.Treeview(self.root, columns=columns, show="headings", height=12)

        self.tree.heading("passenger_id", text="Passenger ID")
        self.tree.heading("Personal_name", text="First Name")
        self.tree.heading("Family_name", text="Family Name")
        self.tree.heading("Age", text="Age")
        self.tree.heading("Sex", text="Sex")

        self.tree.column("passenger_id", width=90, anchor="center")
        self.tree.column("Personal_name", width=140, anchor="center")
        self.tree.column("Family_name", width=140, anchor="center")
        self.tree.column("Age", width=80, anchor="center")
        self.tree.column("Sex", width=80, anchor="center")

        self.tree.pack(fill="both", expand=True, padx=15, pady=10)

        # scrollbar
        scrollbar = ttk.Scrollbar(self.root, orient="vertical", command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        #scrollbar.place(x=715, y=115, height=255)

    def load_cabins(self):
        """
        Load cabins into the dropdown.
        You can change the display text here if needed.
        """
        query = """
        SELECT cabin_id, class, ship_level
        FROM cabin_inventory
        ORDER BY cabin_id
        """
        df = pd.read_sql(query, self.conn)

        # Store display text mapped to cabin_id
        self.cabin_map = {}
        display_values = []

        for _, row in df.iterrows():
            display_text = f"Cabin {row['cabin_id']} - {row['class']} - {row['ship_level']}"
            display_values.append(display_text)
            self.cabin_map[display_text] = row["cabin_id"]

        self.cabin_combo["values"] = display_values

        if display_values:
            self.cabin_combo.current(0)

    def show_passengers(self):
        selected_cabin = self.cabin_combo.get()

        if not selected_cabin:
            messagebox.showwarning("No Selection", "Please choose a cabin first.")
            return

        cabin_id = self.cabin_map[selected_cabin]

        # Get cabin details
        cabin_query = """
        SELECT cabin_id, class, no_of_beds, price, ship_level
        FROM cabin_inventory
        WHERE cabin_id = ?
        """
        cabin_df = pd.read_sql(cabin_query, self.conn, params=(cabin_id,))

        if cabin_df.empty:
            messagebox.showerror("Error", "Cabin not found.")
            return

        cabin = cabin_df.iloc[0]
        self.details_label.config(
            text=(
                f"Cabin {cabin['cabin_id']} | "
                f"Class: {cabin['class']} | "
                f"Deck: {cabin['ship_level']} | "
                f"Beds: {cabin['no_of_beds']} | "
                f"Price: {cabin['price']}"
            )
        )

        # Get passengers in selected cabin
        passenger_query = """
        SELECT passenger_id, Personal_name, Family_name, Age, Sex
        FROM passenger
        WHERE cabin_id = ?
        ORDER BY Family_name, Personal_name
        """
        passenger_df = pd.read_sql(passenger_query, self.conn, params=(cabin_id,))
        print(passenger_df)
        # Clear old rows
        for item in self.tree.get_children():
            self.tree.delete(item)

        # Insert new rows
        
        if passenger_df.empty:
            messagebox.showinfo("No Passengers", "There are no passengers assigned to this cabin.")
        else:
            for _, row in passenger_df.iterrows():
                self.tree.insert(
                    "",
                    "end",
                    values=(
                        row["passenger_id"],
                        row["Personal_name"],
                        row["Family_name"],
                        row["Age"],
                        row["Sex"]
                    )
                )
        
    def close_app(self):
        self.conn.close()
        self.root.destroy()


# Run the program
if __name__ == "__main__":
    CabinPassengerViewer("example.db")